In [4]:
import librosa
import numpy as np
from pathlib import Path
from scipy.spatial.distance import cosine
# dtw-python and MCD are less directly useful here for "similarity" if sentences are different,
# but we can include MCD for completeness, with caveats.
from dtw import dtw

# --- Configuration (REPLACE THESE WITH YOUR FILE PATHS) ---
ORIGINAL_AUDIO_PATH_SAMPLES = [
    "my/cloning/t1.wav"
    # Add more samples of the original voice if available
]

CLONED_AUDIO_PATH_SAMPLES = [
    "cloned_cf950cc6-25fd-40d6-9cd8-4e329b566e7f.wav"
    # Add more samples of the cloned voice saying different things
]

# --- Global Settings ---
SAMPLING_RATE = 16000  # Resample audio to this rate for consistency

# --- Helper: Load and Preprocess Audio for Resemblyzer ---
def preprocess_audio_for_resemblyzer(audio_path_str: str):
    audio_path = Path(audio_path_str)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path_str}")
    from resemblyzer import preprocess_wav
    return preprocess_wav(audio_path)

# --- Metric 1: Speaker Embedding Similarity ---
def get_speaker_embedding(audio_path: str, encoder):
    """Generates a speaker embedding for a single audio file."""
    try:
        wav = preprocess_audio_for_resemblyzer(audio_path)
        embedding = encoder.embed_utterance(wav)
        return embedding
    except FileNotFoundError:
        print(f"File not found: {audio_path}")
        return None
    except Exception as e:
        print(f"Error processing {audio_path} for embedding: {e}")
        return None

def calculate_average_speaker_similarity(original_samples_paths: list, cloned_samples_paths: list):
    """
    Calculates speaker similarity using Resemblyzer embeddings.
    Compares average embedding of original voice to each cloned sample,
    and also average of cloned to each original for a more robust measure.
    """
    try:
        from resemblyzer import VoiceEncoder
        encoder = VoiceEncoder(device='cpu') # Use 'cuda' if available and preferred

        original_embeddings = [get_speaker_embedding(p, encoder) for p in original_samples_paths]
        cloned_embeddings = [get_speaker_embedding(p, encoder) for p in cloned_samples_paths]

        original_embeddings = [e for e in original_embeddings if e is not None]
        cloned_embeddings = [e for e in cloned_embeddings if e is not None]

        if not original_embeddings or not cloned_embeddings:
            print("Not enough valid embeddings to calculate similarity.")
            return None, None

        # Create an average embedding for the original speaker
        avg_original_embedding = np.mean(original_embeddings, axis=0)

        # Compare average original embedding to each cloned sample
        similarities_orig_to_cloned = []
        for clone_emb in cloned_embeddings:
            sim = 1 - cosine(avg_original_embedding, clone_emb)
            similarities_orig_to_cloned.append(sim)
        
        avg_sim_orig_to_cloned = np.mean(similarities_orig_to_cloned) if similarities_orig_to_cloned else None

        # Optional: Create an average embedding for the cloned speaker (if desired)
        # avg_cloned_embedding = np.mean(cloned_embeddings, axis=0)
        # avg_sim_cloned_to_orig = []
        # for orig_emb in original_embeddings:
        #     sim = 1 - cosine(avg_cloned_embedding, orig_emb)
        #     avg_sim_cloned_to_orig.append(sim)
        # overall_avg_similarity = np.mean([avg_sim_orig_to_cloned, np.mean(avg_sim_cloned_to_orig)])

        return avg_sim_orig_to_cloned, similarities_orig_to_cloned

    except Exception as e:
        print(f"Error in Speaker Similarity calculation: {e}")
        print("Ensure ffmpeg is installed and in PATH if Resemblyzer has trouble loading audio.")
        print("Ensure audio files are long enough for Resemblyzer (e.g., > 1-2 seconds).")
        return None, None

# --- (Optional) Metric 2: Mel-Cepstral Distortion (MCD) ---
# NOTE: MCD is less meaningful for similarity if sentences are very different.
# It measures spectral difference, which will naturally be high.
# We include it for completeness but with strong caveats.
def calculate_mcd_between_different_sentences(original_audio_path: str, cloned_audio_path: str, n_mfcc=13):
    """Calculates MCD. High values expected if content differs."""
    try:
        y_orig, sr_orig = librosa.load(original_audio_path, sr=SAMPLING_RATE)
        y_cloned, sr_cloned = librosa.load(cloned_audio_path, sr=SAMPLING_RATE)

        mfcc_orig = librosa.feature.mfcc(y=y_orig, sr=SAMPLING_RATE, n_mfcc=n_mfcc)
        mfcc_cloned = librosa.feature.mfcc(y=y_cloned, sr=SAMPLING_RATE, n_mfcc=n_mfcc)

        alignment = dtw(mfcc_orig.T, mfcc_cloned.T, keep_internals=False, distance_only=False)
        path_original = alignment.index1
        path_cloned = alignment.index2
        
        aligned_original_mfcc = mfcc_orig[:, path_original]
        aligned_cloned_mfcc = mfcc_cloned[:, path_cloned]
        
        mcd_val = np.mean(np.sqrt(np.sum((aligned_original_mfcc - aligned_cloned_mfcc)**2, axis=0)))
        return mcd_val
    except FileNotFoundError:
        print(f"Error: MCD requires both audio files. Check paths.")
        return None
    except Exception as e:
        print(f"Error in MCD calculation: {e}")
        return None

# --- Main Execution ---
if __name__ == "__main__":
    if not ORIGINAL_AUDIO_PATH_SAMPLES or not CLONED_AUDIO_PATH_SAMPLES:
        print("Please provide at least one original and one cloned audio sample path.")
    else:
        print("Comparing Original Voice Samples vs. Cloned Voice Samples (Text-Independent)\n")

        # 1. Speaker Embedding Similarity
        print("--- 1. Speaker Embedding Similarity ---")
        avg_similarity, individual_similarities = calculate_average_speaker_similarity(
            ORIGINAL_AUDIO_PATH_SAMPLES,
            CLONED_AUDIO_PATH_SAMPLES
        )

        if avg_similarity is not None:
            print(f"Average Cosine Similarity (Original Avg Emb to Cloned Samples): {avg_similarity:.4f}")
            print("(Higher is more similar, closer to 1.0 means more identical vocal characteristics)")
            if individual_similarities:
                 print("Individual Similarities (Original Avg Emb to each Cloned Sample):")
                 for i, sim in enumerate(individual_similarities):
                     print(f"  Cloned Sample {i+1} ('{CLONED_AUDIO_PATH_SAMPLES[i]}'): {sim:.4f}")
            print("\n")
        else:
            print("Could not calculate speaker similarity.\n")

        # (Optional) MCD - Calculate for the first pair as an example, with caveats
        #print("--- (Optional) Mel-Cepstral Distortion (MCD) ---")
        #print("NOTE: MCD measures spectral difference. If sentences are different, a higher MCD is expected.")
        #print("      It's less a measure of 'voice identity similarity' in this context and more a measure")
        #print("      of raw acoustic difference between the specific utterances.")

        #if ORIGINAL_AUDIO_PATH_SAMPLES and CLONED_AUDIO_PATH_SAMPLES:
        #   mcd_score = calculate_mcd_between_different_sentences(
        #      ORIGINAL_AUDIO_PATH_SAMPLES[0],
        #     CLONED_AUDIO_PATH_SAMPLES[0])
            #if mcd_score is not None:
                #print(f"MCD Score (between '{ORIGINAL_AUDIO_PATH_SAMPLES[0]}' and '{CLONED_AUDIO_PATH_SAMPLES[0]}'): {mcd_score:.4f}")
                #print("(Lower means spectrally closer for these specific utterances, but high values are expected if content differs)\n")
            #else:
                #print("Could not calculate MCD for the sample pair.\n")
        #else:
            #print("Not enough samples to calculate example MCD.\n")


       

Comparing Original Voice Samples vs. Cloned Voice Samples (Text-Independent)

--- 1. Speaker Embedding Similarity ---
Loaded the voice encoder model on cpu in 0.21 seconds.
Average Cosine Similarity (Original Avg Emb to Cloned Samples): 0.5902
(Higher is more similar, closer to 1.0 means more identical vocal characteristics)
Individual Similarities (Original Avg Emb to each Cloned Sample):
  Cloned Sample 1 ('cloned_cf950cc6-25fd-40d6-9cd8-4e329b566e7f.wav'): 0.5902




In [2]:
!pip install -r requirements1.txt


  Using cached librosa-0.10.1-py3-none-any.whl.metadata (8.3 kB)
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
  Using cached soxr-0.5.0.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
  Using cached msgpack-1.1.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 163.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 154.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.7 MB/s eta 0:00:00
Using cached audioread-3.0.1-py3-none-any.